# 07 — Memory Quantile Prediction Model

Parallel to notebook 03 (CPU model), this trains XGBoost quantile regression on
memory utilization to predict future memory usage bounds for each VM.

**Key difference from the CPU model:** we lack daily memory time series, so we
cannot do the temporal split (days 1-25 features, days 26-31 target). Instead we
train on summary-derived features predicting `mem_mean` — the same features the
model will see at inference time in `predict.py`.

**Features** (5, matching CPU model pattern after VIF cleaning):
- `mem_std` — approximated from `(mem_max - mem_min) / 4`
- `mem_min` — minimum observed memory utilization
- `mem_p50` — median memory utilization
- `mem_trend` — slope over time (0.0 without daily series)
- `mem_cv` — coefficient of variation (std / mean)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid", palette="muted")

sys.path.insert(0, "..")
from src.memory_model import build_memory_features, MEM_FEATURE_NAMES

DATA = Path("../data/clean")
MODELS = Path("../models")
FIGURES = Path("../reports/figures")
FIGURES.mkdir(parents=True, exist_ok=True)

## 1. Load and explore memory data

In [ ]:
csv_path = DATA / "vm_utilization_summary.csv"
instances, X = build_memory_features(csv_path)

print(f"VMs with memory data: {len(instances):,}")
print(f"Features: {MEM_FEATURE_NAMES}")
print(f"Shape: {X.shape}")
print()

df_feat = pd.DataFrame(X, columns=MEM_FEATURE_NAMES)
df_feat.describe().round(2)

In [ ]:
# Load target: mem_mean
summary = pd.read_csv(csv_path)
summary = summary[summary["mem_n"] > 0].reset_index(drop=True)
y = summary["mem_mean"].values

print(f"Target (mem_mean) stats:")
print(f"  Mean:   {np.mean(y):.2f}%")
print(f"  Median: {np.median(y):.2f}%")
print(f"  Std:    {np.std(y):.2f}%")
print(f"  Range:  {np.min(y):.2f}% - {np.max(y):.2f}%")

## 2. Feature correlation and VIF analysis

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
corr = df_feat.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, ax=ax, linewidths=0.5)
ax.set_title("Memory feature correlation matrix")
plt.savefig(FIGURES / "mem_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def compute_vif(X_df):
    return pd.DataFrame({
        "feature": X_df.columns,
        "VIF": [variance_inflation_factor(X_df.values, i) for i in range(X_df.shape[1])]
    }).sort_values("VIF", ascending=False).reset_index(drop=True)

vif = compute_vif(df_feat)
print("VIF analysis for memory features:")
print(vif.to_string(index=False))
print()
if (vif["VIF"] > 10).any():
    print("WARNING: some features have VIF > 10 — consider dropping")
else:
    print("All features have VIF < 10 — no multicollinearity issues")

## 3. Train/test split

In [ ]:
X_train, X_test, y_train, y_test, inst_train, inst_test = train_test_split(
    X, y, np.array(instances), test_size=0.20, random_state=42
)

print(f"Train VMs: {len(X_train):,}")
print(f"Test VMs:  {len(X_test):,}")

## 4. Naive baseline

In [ ]:
# p50 is the 3rd feature (index 2): mem_std, mem_min, mem_p50, mem_trend, mem_cv
naive_preds = X_test[:, 2]  # mem_p50
naive_mae = mean_absolute_error(y_test, naive_preds)
naive_under_rate = np.mean(naive_preds < y_test) * 100

print(f"Naive baseline (predict mem_p50): MAE = {naive_mae:.2f}%")
print(f"Naive underprediction rate:       {naive_under_rate:.1f}%")

## 5. Train XGBoost quantile models

Same architecture as the CPU model:
- q=0.10 (optimistic lower bound)
- q=0.50 with asymmetric loss (3x underprediction penalty)
- q=0.95 (conservative upper bound for decisions)

In [ ]:
import yaml

with open("../config.yaml") as f:
    config = yaml.safe_load(f)

model_cfg = config["model"]
UNDER_PENALTY = model_cfg["under_penalty"]

def asymmetric_squared_error(y_true, y_pred):
    residual = y_true - y_pred
    grad = np.where(residual > 0, -2 * UNDER_PENALTY * residual, -2 * residual)
    hess = np.where(residual > 0, 2 * UNDER_PENALTY, 2.0) * np.ones_like(residual)
    return grad, hess

quantiles = model_cfg["quantiles"]
models = {}

for q in quantiles:
    print(f"Training q={q}...")

    if q == 0.50:
        model = xgb.XGBRegressor(
            objective=asymmetric_squared_error,
            n_estimators=model_cfg["n_estimators"],
            max_depth=model_cfg["max_depth"],
            learning_rate=model_cfg["learning_rate"],
            min_child_weight=model_cfg["min_child_weight"],
            subsample=model_cfg["subsample"],
            colsample_bytree=model_cfg["colsample_bytree"],
            reg_lambda=model_cfg["reg_lambda"],
            random_state=config["seed"],
            early_stopping_rounds=model_cfg["early_stopping_rounds"],
            verbosity=0,
        )
    else:
        model = xgb.XGBRegressor(
            objective="reg:quantileerror",
            quantile_alpha=q,
            n_estimators=model_cfg["n_estimators"],
            max_depth=model_cfg["max_depth"],
            learning_rate=model_cfg["learning_rate"],
            min_child_weight=model_cfg["min_child_weight"],
            subsample=model_cfg["subsample"],
            colsample_bytree=model_cfg["colsample_bytree"],
            reg_lambda=model_cfg["reg_lambda"],
            random_state=config["seed"],
            early_stopping_rounds=model_cfg["early_stopping_rounds"],
            verbosity=0,
        )

    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)
    train_mae = mean_absolute_error(y_train, pred_train)
    test_mae = mean_absolute_error(y_test, pred_test)
    under_rate = np.mean(pred_test < y_test) * 100

    label = f"q={q}" if q != 0.50 else f"q={q} (asymmetric, {UNDER_PENALTY:.0f}x penalty)"
    print(f"  {label}")
    print(f"  Best iteration: {model.best_iteration} / {model_cfg['n_estimators']}")
    print(f"  Train MAE: {train_mae:.2f}%  |  Test MAE: {test_mae:.2f}%")
    print(f"  Gap: {test_mae - train_mae:.2f}%  |  Underprediction rate: {under_rate:.1f}%")
    print()
    models[q] = model

## 6. Evaluate predictions

In [ ]:
preds_test = {q: models[q].predict(X_test) for q in quantiles}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, q in zip(axes, quantiles):
    label = f"q={q}" if q != 0.50 else f"q={q} (asymmetric)"
    ax.scatter(y_test, preds_test[q], alpha=0.05, s=3)
    ax.plot([0, 100], [0, 100], "r--", alpha=0.5)
    ax.set_xlabel("Actual mean memory (%)")
    ax.set_ylabel(f"Predicted {label}")
    ax.set_title(f"{label} — Test MAE: {mean_absolute_error(y_test, preds_test[q]):.2f}%")
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)

plt.tight_layout()
plt.savefig(FIGURES / "mem_model_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
coverage_85 = np.mean((y_test >= preds_test[0.10]) & (y_test <= preds_test[0.95]))
print(f"85% prediction interval coverage (q=0.10 to q=0.95, test set): {coverage_85*100:.1f}%")
print("  -> target: 85%")

## 7. Feature importance

In [ ]:
importances = models[0.50].feature_importances_
feat_imp = pd.DataFrame({"feature": MEM_FEATURE_NAMES, "importance": importances})
feat_imp = feat_imp.sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(feat_imp["feature"], feat_imp["importance"], color="#94A3B8", edgecolor="white")
ax.set_xlabel("Feature importance (gain)")
ax.set_title("Memory model — what drives the prediction?")
ax.invert_yaxis()
plt.savefig(FIGURES / "mem_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    models[0.50], X_test, y_test,
    n_repeats=10, scoring="neg_mean_absolute_error",
    random_state=42, n_jobs=-1,
)

perm_imp = pd.DataFrame({
    "feature": MEM_FEATURE_NAMES,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std,
}).sort_values("importance_mean", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.barh(feat_imp["feature"], feat_imp["importance"], color="#94A3B8", edgecolor="white")
ax.set_xlabel("Importance (gain)")
ax.set_title("Gain-based importance")
ax.invert_yaxis()

ax = axes[1]
ax.barh(perm_imp["feature"], perm_imp["importance_mean"],
        xerr=perm_imp["importance_std"], color="#60A5FA", edgecolor="white", capsize=3)
ax.set_xlabel("MAE increase when shuffled")
ax.set_title("Permutation importance (10 repeats)")
ax.invert_yaxis()

plt.suptitle("Memory model feature importance: gain vs permutation", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES / "mem_permutation_importance.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nPermutation importance ranking:")
for _, row in perm_imp.iterrows():
    print(f"  {row['feature']:>10}: {row['importance_mean']:.4f} ± {row['importance_std']:.4f}")

## 8. Save memory models

In [ ]:
for q, model in models.items():
    path = MODELS / f"mem_quantile_{q:.2f}.json"
    model.save_model(path)
    print(f"Saved {path} (best iteration: {model.best_iteration})")

print(f"\nMemory models saved to {MODELS}/")

## 9. Model summary

| Metric | Value |
|---|---|
| Algorithm | XGBoost quantile regression (q=0.10, 0.50, 0.95) |
| Median model loss | Asymmetric (3x penalty on underpredictions) |
| Features | 5 summary-derived (mem_std, mem_min, mem_p50, mem_trend, mem_cv) |
| Split | 80% train / 20% test (random, no temporal split — see note) |

**Note on temporal split:** The CPU model uses a temporal split (days 1-25 features,
days 26-31 target) because daily CPU time series data is available. For memory,
only summary statistics exist in the dataset, so we use a random VM-level split.
When daily memory data becomes available (via `ingest.py` re-run with raw data),
this model should be retrained with a proper temporal split.